# 06: Report Generation — Complete Chart Gallery

This notebook demonstrates every chart type available in `siege_utilities.reporting.ChartGenerator`,
plus the `ChartTypeRegistry` extension system and `PollingAnalyzer` analytics engine.

**Sections:**

1. [Imports & Setup](#1-imports--setup)
2. [Standard Charts](#2-standard-charts) — bar, line, pie, scatter, heatmap
3. [Geographic Maps (Folium)](#3-geographic-maps-folium) — marker, heatmap, cluster, flow
4. [Geographic Maps (Matplotlib)](#4-geographic-maps-matplotlib) — advanced choropleth, bivariate choropleth
5. [3D Visualization](#5-3d-visualization) — 3D scatter/surface
6. [Isochrone Service Areas](#5b-isochrone-service-areas) — open-source provider integration
7. [Text-Based Charts](#6-text-based-charts) — proportional bar, heatmap text
8. [Composite Charts & Layout](#7-composite-charts--layout) — dashboard, summary, custom, dispatcher
9. [Complete PDF Report](#8-complete-pdf-report) — multi-page PDF with chart_with_caption, chart_section
10. [PowerPoint Generation](#9-powerpoint-generation)
11. [Chart Type Registry](#10-chart-type-registry) — extensible chart catalogue, custom types, validation
12. [Polling / Analytics](#11-polling--analytics) — cross-tabulation, longitudinal analysis, rankings

## Prerequisites

```bash
pip install -e /path/to/siege_utilities[geo]
pip install reportlab python-pptx pillow folium
```


In [ ]:
# 1. Imports & Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage
import io
import base64

import siege_utilities as su
su.configure_shared_logging(level="INFO")

from siege_utilities.reporting.chart_generator import ChartGenerator
from siege_utilities.reporting.image_utils import show_rl_image, save_rl_image
from reportlab.platypus import Image as RLImage

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_06"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Branding configuration ---
from siege_utilities.reporting.client_branding import ClientBrandingManager
_branding_mgr = ClientBrandingManager()
BRANDING = _branding_mgr.get_client_branding('siege_analytics')
COLORS = BRANDING['colors']

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")
su.log_info(f"Branding: {BRANDING['name']} (primary={COLORS['primary']})")

# Screen / notebook display — no size limits
chart_gen = ChartGenerator(branding_config=BRANDING, output_dir=OUTPUT_DIR)

# PDF-safe variant — auto-scales oversized charts to fit letter page frame
# (456pt / 6.33in wide, 636pt / 8.83in tall with 1-inch margins)
chart_gen_pdf = ChartGenerator(
    branding_config=BRANDING, output_dir=OUTPUT_DIR,
    max_chart_width=6.0, max_chart_height=8.5,
)

np.random.seed(42)

su.log_info(f"ChartGenerator initialized with {BRANDING['name']} branding")
su.log_info(f"Available create_* methods: {len([m for m in dir(chart_gen) if m.startswith('create_')])}")
su.log_info(f"PDF chart generator: max_chart_width={chart_gen_pdf.max_chart_width}, max_chart_height={chart_gen_pdf.max_chart_height}")

In [ ]:
# Sample Data — used throughout the notebook

# 1. Sales data (bar, line, pie, dashboard, PDF)
sales_data = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'],
    'Revenue': [45000, 52000, 48000, 61000, 55000, 67000],
    'Expenses': [32000, 35000, 33000, 38000, 36000, 40000],
    'Units': [150, 175, 160, 205, 185, 225],
})
sales_data['Profit'] = sales_data['Revenue'] - sales_data['Expenses']

# 2. Regional data (pie, scatter, text charts)
regional_data = pd.DataFrame({
    'Region': ['North', 'South', 'East', 'West', 'Central'],
    'Sales': [125000, 98000, 145000, 112000, 89000],
    'Customers': [450, 380, 520, 410, 340],
    'Avg Order': [278, 258, 279, 273, 262],
})

# 3. Point data — US cities (marker map, heatmap map, cluster map, 3D)
cities = [
    ('New York', 40.71, -74.01), ('Los Angeles', 34.05, -118.24),
    ('Chicago', 41.88, -87.63), ('Houston', 29.76, -95.37),
    ('Phoenix', 33.45, -112.07), ('Philadelphia', 39.95, -75.17),
    ('San Antonio', 29.42, -98.49), ('San Diego', 32.72, -117.16),
    ('Dallas', 32.78, -96.80), ('San Jose', 37.34, -121.89),
    ('Austin', 30.27, -97.74), ('Jacksonville', 30.33, -81.66),
    ('San Francisco', 37.77, -122.42), ('Columbus', 39.96, -82.99),
    ('Indianapolis', 39.77, -86.16), ('Fort Worth', 32.76, -97.33),
    ('Charlotte', 35.23, -80.84), ('Seattle', 47.61, -122.33),
    ('Denver', 39.74, -104.98), ('Nashville', 36.16, -86.78),
]
point_data = pd.DataFrame(cities, columns=['name', 'lat', 'lon'])
point_data['value'] = np.random.randint(100, 5000, size=len(point_data))
point_data['category'] = np.random.choice(['Tech', 'Finance', 'Healthcare'], size=len(point_data))

# 4. Flow data — origin/destination pairs
flow_data = pd.DataFrame({
    'origin_lat':  [40.71, 34.05, 41.88, 29.76, 47.61, 39.74, 37.77, 33.45],
    'origin_lon':  [-74.01, -118.24, -87.63, -95.37, -122.33, -104.98, -122.42, -112.07],
    'dest_lat':    [34.05, 41.88, 29.76, 33.45, 37.77, 40.71, 47.61, 39.74],
    'dest_lon':    [-118.24, -87.63, -95.37, -112.07, -122.42, -74.01, -122.33, -104.98],
    'flow_value':  [1500, 2200, 800, 1100, 1800, 950, 1300, 600],
})

# 5. Correlation data (heatmap, scatter, summary charts)
n = 30
correlation_data = pd.DataFrame({
    'x': np.random.randn(n) * 10 + 50,
    'y': np.random.randn(n) * 15 + 70,
    'z': np.random.randn(n) * 5 + 20,
    'w': np.random.randn(n) * 8 + 40,
    'v': np.random.randn(n) * 12 + 60,
})

# 6. Heatmap text data — quarterly sales by region
heatmap_text_data = pd.DataFrame([
    {'quarter': 'Q1', 'region': 'North', 'sales': 120},
    {'quarter': 'Q1', 'region': 'South', 'sales': 95},
    {'quarter': 'Q1', 'region': 'East', 'sales': 140},
    {'quarter': 'Q1', 'region': 'West', 'sales': 110},
    {'quarter': 'Q2', 'region': 'North', 'sales': 135},
    {'quarter': 'Q2', 'region': 'South', 'sales': 105},
    {'quarter': 'Q2', 'region': 'East', 'sales': 155},
    {'quarter': 'Q2', 'region': 'West', 'sales': 125},
    {'quarter': 'Q3', 'region': 'North', 'sales': 150},
    {'quarter': 'Q3', 'region': 'South', 'sales': 115},
    {'quarter': 'Q3', 'region': 'East', 'sales': 165},
    {'quarter': 'Q3', 'region': 'West', 'sales': 130},
    {'quarter': 'Q4', 'region': 'North', 'sales': 160},
    {'quarter': 'Q4', 'region': 'South', 'sales': 125},
    {'quarter': 'Q4', 'region': 'East', 'sales': 180},
    {'quarter': 'Q4', 'region': 'West', 'sales': 145},
])

su.log_info(f"sales_data:       {sales_data.shape}")
su.log_info(f"regional_data:    {regional_data.shape}")
su.log_info(f"point_data:       {point_data.shape}")
su.log_info(f"flow_data:        {flow_data.shape}")
su.log_info(f"correlation_data: {correlation_data.shape}")
su.log_info(f"heatmap_text_data:{heatmap_text_data.shape}")

## 2. Standard Charts

Matplotlib-backed charts that return ReportLab `Image` objects.  We use `show_rl_image()` to
decode the base64 data URI for notebook display.

In [ ]:
# 2a. Bar Chart — vertical
img = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue',
)
save_rl_image(img, OUTPUT_DIR / 'bar_vertical.png')
display(show_rl_image(img))

# 2a. Bar Chart — horizontal
img_h = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue (Horizontal)',
    chart_type='horizontal',
)
save_rl_image(img_h, OUTPUT_DIR / 'bar_horizontal.png')
display(show_rl_image(img_h))

In [ ]:
# 2b. Line Chart — multi-series
img = chart_gen.create_line_chart(
    sales_data, x_column='Month',
    y_columns=['Revenue', 'Expenses'],
    title='Revenue vs Expenses',
)
save_rl_image(img, OUTPUT_DIR / 'line_revenue_expenses.png')
display(show_rl_image(img))

In [ ]:
# 2c. Pie Chart — with legend table
img = chart_gen.create_pie_chart(
    regional_data,
    labels_column='Region', values_column='Sales',
    title='Sales by Region',
)
save_rl_image(img, OUTPUT_DIR / 'pie_regional_sales.png')
display(show_rl_image(img))

In [ ]:
# 2d. Scatter Plot — with color coding
img = chart_gen.create_scatter_plot(
    correlation_data,
    x_column='x', y_column='y', color_column='z',
    title='Scatter with Color Coding',
)
save_rl_image(img, OUTPUT_DIR / 'scatter_color_coded.png')
display(show_rl_image(img))

In [ ]:
# 2e. Heatmap — correlation matrix
img = chart_gen.create_heatmap(
    correlation_data,
    title='Correlation Matrix',
)
save_rl_image(img, OUTPUT_DIR / 'heatmap_correlation.png')
display(show_rl_image(img))

## 3. Geographic Maps (Folium)

Folium-based methods save HTML files and return placeholder ReportLab images.
We call the ChartGenerator API to exercise it, then also create Folium map objects
directly so we get interactive output in the notebook.

In [ ]:
# 3a. Marker Map
import folium
from folium import plugins

# Call ChartGenerator API (saves HTML, returns placeholder)
placeholder = chart_gen.create_marker_map(
    point_data, latitude_column='lat', longitude_column='lon',
    value_column='value', label_column='name',
    title='US Cities - Marker Map', zoom_level=4,
)
su.log_info("create_marker_map() returned placeholder image")

# Display interactive Folium map inline
m = folium.Map(location=[39.0, -98.0], zoom_start=4)
for _, row in point_data.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(5, row['value'] / 200),
        popup=f"<b>{row['name']}</b><br>Value: {row['value']:,}",
        color='red', fill=True, fill_opacity=0.7,
    ).add_to(m)
display(m)

In [ ]:
# 3b. Heatmap Map
placeholder = chart_gen.create_heatmap_map(
    point_data, latitude_column='lat', longitude_column='lon',
    value_column='value',
    title='US Cities - Heatmap',
)
su.log_info("create_heatmap_map() returned placeholder image")

# Display interactive heatmap
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
heat_data = [[row['lat'], row['lon'], row['value']] for _, row in point_data.iterrows()]
plugins.HeatMap(heat_data, radius=25, blur=15).add_to(m)
display(m)

In [ ]:
# 3c. Cluster Map
placeholder = chart_gen.create_cluster_map(
    point_data, latitude_column='lat', longitude_column='lon',
    label_column='name',
    title='US Cities - Cluster Map',
)
su.log_info("create_cluster_map() returned placeholder image")

# Display interactive cluster map
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
mc = plugins.MarkerCluster()
for _, row in point_data.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"<b>{row['name']}</b><br>Category: {row['category']}",
    ).add_to(mc)
mc.add_to(m)
display(m)

In [ ]:
# 3d. Flow Map
placeholder = chart_gen.create_flow_map(
    flow_data,
    origin_lat_column='origin_lat', origin_lon_column='origin_lon',
    dest_lat_column='dest_lat', dest_lon_column='dest_lon',
    flow_value_column='flow_value',
    title='US City Flows',
)
su.log_info("create_flow_map() returned placeholder image")

# Display interactive flow map
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
max_flow = flow_data['flow_value'].max()
for _, row in flow_data.iterrows():
    weight = max(1, int(row['flow_value'] / max_flow * 8))
    folium.PolyLine(
        locations=[
            [row['origin_lat'], row['origin_lon']],
            [row['dest_lat'], row['dest_lon']],
        ],
        weight=weight, color='red', opacity=0.6,
        popup=f"Flow: {row['flow_value']:,}",
    ).add_to(m)
    folium.CircleMarker([row['origin_lat'], row['origin_lon']], radius=4, color='blue', fill=True).add_to(m)
    folium.CircleMarker([row['dest_lat'], row['dest_lon']], radius=4, color='green', fill=True).add_to(m)
display(m)

## 4. Geographic Maps (Matplotlib & Folium Choropleth)

These methods require a GeoDataFrame with geometry. We load California counties using the
same `get_census_boundaries()` pattern from NB05, then use them for both Folium choropleth
maps and matplotlib-rendered maps.

In [ ]:
# Load CA county boundaries
# get_census_boundaries() caches files via its own internal mechanism.
from siege_utilities.geo import get_census_boundaries

counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')

if counties is None:
    su.log_warning(f"get_census_boundaries() returned None.")
    su.log_warning("Retrying with year=2021...")
    counties = get_census_boundaries(year=2021, geographic_level='county', state_fips='06')

if counties is None:
    raise RuntimeError(
        f"Could not download Census county boundaries.\n"
        f"  Check network connectivity and try again, or run NB05 first to cache the shapefile."
    )

ca_counties = counties[counties['statefp'] == '06'].copy()

# Simulate data columns
ca_counties['population'] = np.random.randint(10000, 10000000, size=len(ca_counties))
ca_counties['median_income'] = np.random.randint(40000, 150000, size=len(ca_counties))

su.log_info(f"Loaded {len(ca_counties)} California counties")
su.log_info(f"Columns: {list(ca_counties.columns)[:8]}...")

In [ ]:
# 4a. Choropleth Map (Folium) — uses get_census_boundaries() GeoDataFrame
# Prepare data for Folium choropleth — needs a plain DataFrame + separate GeoJSON
choropleth_data = ca_counties[['geoid', 'population']].copy()

placeholder = chart_gen.create_choropleth_map(
    choropleth_data, geo_data=ca_counties,
    location_column='geoid', value_column='population',
    title='CA Population Choropleth (Folium)',
    map_type='us',
    key_on='feature.properties.geoid',
    fill_color='YlOrRd',
)
su.log_info("create_choropleth_map() — HTML saved, placeholder returned")

# Display interactive Folium map inline
import json
m = folium.Map(location=[37.0, -120.0], zoom_start=6, tiles='cartodbpositron')
folium.Choropleth(
    geo_data=json.loads(ca_counties.to_json()),
    data=choropleth_data,
    columns=['geoid', 'population'],
    key_on='feature.properties.geoid',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Population',
).add_to(m)
display(m)

In [ ]:
# 4b. Bivariate Choropleth (Folium) — interactive version of the matplotlib bivariate
placeholder = chart_gen.create_bivariate_choropleth(
    ca_counties, geo_data=ca_counties,
    location_column='geoid',
    value_column1='population', value_column2='median_income',
    title='Population vs Income (Folium)',
    color_scheme='default',
)
su.log_info("create_bivariate_choropleth() — HTML saved, placeholder returned")

# The saved HTML at OUTPUT_DIR/temp_bivariate_choropleth.html is interactive.
# Display the saved map inline:
from IPython.display import IFrame
bivar_path = OUTPUT_DIR / "temp_bivariate_choropleth.html"
if bivar_path.exists():
    display(IFrame(src=str(bivar_path), width=800, height=500))

In [ ]:
# 4c. Advanced Choropleth (matplotlib)
img = chart_gen.create_advanced_choropleth(
    ca_counties, geodata=ca_counties,
    location_column='geoid', value_column='population',
    title='CA Population by County (Quantiles)',
    classification='quantiles', bins=5,
)
save_rl_image(img, OUTPUT_DIR / 'choropleth_quantiles.png')
display(show_rl_image(img))

In [ ]:
# 4d. Bivariate Choropleth (matplotlib)
img = chart_gen.create_bivariate_choropleth_matplotlib(
    ca_counties, geodata=ca_counties,
    location_column='geoid',
    value_column1='population', value_column2='median_income',
    title='Population vs Income (matplotlib)',
)
save_rl_image(img, OUTPUT_DIR / 'bivariate_matplotlib.png')
display(show_rl_image(img))

## 5. 3D Visualization

Uses matplotlib's 3D projection for point-cloud and surface plots.

In [ ]:
# 5a. 3D Map — point cloud with simulated elevation
img = chart_gen.create_3d_map(
    point_data,
    latitude_column='lat', longitude_column='lon',
    elevation_column='value',
    title='3D US City Point Cloud',
)
save_rl_image(img, OUTPUT_DIR / '3d_point_cloud.png')
display(show_rl_image(img))

## 5b. Isochrone Service Areas

This demonstrates the open-source isochrone API integration (ORS/Valhalla), including custom server support.


In [ ]:
import os
import json
from siege_utilities.geo.isochrones import build_isochrone_request, get_isochrone

# Example center: Chicago
center_lat, center_lon = 41.8781, -87.6298

# Build request definitions without making network calls
ors_req = build_isochrone_request(
    latitude=center_lat,
    longitude=center_lon,
    travel_time_minutes=15,
    provider='openrouteservice',
    profile='driving-car',
)

valhalla_req = build_isochrone_request(
    latitude=center_lat,
    longitude=center_lon,
    travel_time_minutes=20,
    provider='valhalla',
    base_url='http://valhalla.svc.cluster.local:8002',
    profile='driving-car',
)

print('ORS endpoint:', ors_req['url'])
print('Valhalla endpoint:', valhalla_req['url'])

# Optional live request when ORS_API_KEY is available
ors_api_key = os.getenv('ORS_API_KEY')
if ors_api_key:
    try:
        iso = get_isochrone(
            latitude=center_lat,
            longitude=center_lon,
            travel_time_minutes=15,
            provider='openrouteservice',
            api_key=ors_api_key,
            profile='driving-car',
        )
        out_geojson = OUTPUT_DIR / 'isochrone_chicago_15min.geojson'
        with open(out_geojson, 'w', encoding='utf-8') as f:
            json.dump(iso, f)
        print('Saved:', out_geojson)
    except Exception as e:
        print('Live ORS isochrone request failed:', type(e).__name__, str(e))
else:
    print('ORS_API_KEY not set; skipping live isochrone download.')


## 6. Text-Based Charts

These return HTML strings instead of images — useful for email reports, lightweight dashboards,
or environments without matplotlib.

In [ ]:
# 6a. Proportional Text Bar Chart
text_chart = chart_gen.create_proportional_text_bar_chart(
    regional_data,
    labels_column='Region', values_column='Sales',
    title='Sales by Region (Text)',
)
display(HTML(f'<pre style="font-family: monospace; line-height: 1.4;">{text_chart}</pre>'))

In [ ]:
# 6b. Heatmap Text Chart
text_heatmap = chart_gen.create_heatmap_text_chart(
    heatmap_text_data,
    x_column='quarter', y_column='region', value_column='sales',
    title='Quarterly Sales by Region (Text)',
)
display(HTML(f'<pre style="font-family: monospace; line-height: 1.4;">{text_heatmap}</pre>'))

## 7. Composite Charts & Layout

Higher-level methods that combine multiple charts or dispatch by configuration.

In [ ]:
# 7a. Dashboard — 2x2 grid of four chart types
months = sales_data['Month'].tolist()
revenue = sales_data['Revenue'].tolist()
expenses = sales_data['Expenses'].tolist()

dashboard_charts = [
    {
        'type': 'bar',
        'title': 'Revenue by Month',
        'data': {'labels': months, 'datasets': [{'data': revenue}]},
    },
    {
        'type': 'line',
        'title': 'Revenue Trend',
        'data': {'labels': months, 'datasets': [{'data': revenue}]},
    },
    {
        'type': 'pie',
        'title': 'Regional Sales',
        'data': {'labels': regional_data['Region'].tolist(), 'data': regional_data['Sales'].tolist()},
    },
    {
        'type': 'scatter',
        'title': 'X vs Y',
        'data': {'x': correlation_data['x'].tolist(), 'y': correlation_data['y'].tolist()},
    },
]

img = chart_gen.create_dashboard(dashboard_charts, layout='2x2')
save_rl_image(img, OUTPUT_DIR / 'dashboard_2x2.png')
display(show_rl_image(img))

In [ ]:
# 7b. DataFrame Summary Charts — auto-histogram for numeric columns
img = chart_gen.create_dataframe_summary_charts(
    correlation_data,
    title='Data Distributions',
)
save_rl_image(img, OUTPUT_DIR / 'summary_histograms.png')
display(show_rl_image(img))

In [ ]:
# 7c. Custom Chart Dispatcher — create_custom_chart()
config = {
    'type': 'bar',
    'data': sales_data,
    'title': 'Custom Bar Chart via Dispatcher',
}
img = chart_gen.create_custom_chart(config)
save_rl_image(img, OUTPUT_DIR / 'custom_bar.png')
display(show_rl_image(img))

In [ ]:
# 7d. generate_chart_from_dataframe — generic dispatcher
for chart_type in ['bar', 'line', 'pie', 'scatter']:
    su.log_info(f"Generating {chart_type} chart...")
    img = chart_gen.generate_chart_from_dataframe(
        sales_data, chart_type=chart_type,
        x_column='Month', y_columns=['Revenue'],
        title=f'{chart_type.title()} via Dispatcher',
    )
    save_rl_image(img, OUTPUT_DIR / f'dispatch_{chart_type}.png')
    display(show_rl_image(img))

# Heatmap uses correlation_data (all-numeric) — dispatcher computes correlation matrix
su.log_info("Generating heatmap chart...")
img = chart_gen.generate_chart_from_dataframe(
    correlation_data, chart_type='heatmap',
    title='Heatmap via Dispatcher',
)
save_rl_image(img, OUTPUT_DIR / 'dispatch_heatmap.png')
display(show_rl_image(img))

## 8. Complete PDF Report

Build a multi-page PDF demonstrating `BaseReportTemplate`, `create_chart_with_caption()`,
and `create_chart_section()` — these return ReportLab flowable lists that only make sense
inside a PDF build pipeline.

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, HRFlowable
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from siege_utilities.reporting.templates.base_template import BaseReportTemplate

output_pdf = str(OUTPUT_DIR / 'complete_chart_gallery.pdf')

# --- Branding colors as ReportLab color objects ---
brand_primary = colors.HexColor(COLORS['primary'])       # #1D365D dark navy
brand_secondary = colors.HexColor(COLORS['secondary'])   # #4A90E2 bright blue
brand_accent = colors.HexColor(COLORS['accent'])         # #FF6B35 orange
brand_bg = colors.HexColor(COLORS['background'])         # #F6F6F6 light gray

# PDF chart generator — 200 DPI, sized to fit letter page with headings/captions
chart_gen_pdf = ChartGenerator(
    branding_config=BRANDING, output_dir=OUTPUT_DIR,
    max_chart_width=6.0, max_chart_height=5.5,
)
chart_gen_pdf.default_dpi = 200

template = BaseReportTemplate(
    output_filename=output_pdf,
    page_size='letter',
    margins={'top': 1.0, 'bottom': 1.0, 'left': 1.0, 'right': 1.0},
)

# --- Custom branded styles ---
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    'BrandTitle', parent=styles['Title'],
    fontSize=28, textColor=brand_primary, spaceAfter=6,
))
styles.add(ParagraphStyle(
    'BrandSubtitle', parent=styles['Normal'],
    fontSize=14, textColor=brand_secondary, alignment=TA_CENTER, spaceAfter=20,
))
styles.add(ParagraphStyle(
    'BrandHeading', parent=styles['Heading1'],
    fontSize=18, textColor=brand_primary, spaceBefore=0, spaceAfter=8,
))
styles.add(ParagraphStyle(
    'BrandCaption', parent=styles['Normal'],
    fontSize=9, textColor=colors.HexColor('#666666'), italic=True,
    spaceBefore=4, spaceAfter=2,
))
styles.add(ParagraphStyle(
    'BrandMeta', parent=styles['Normal'],
    fontSize=10, textColor=colors.HexColor('#666666'), alignment=TA_CENTER,
))

story = []

# --- Branded title page ---
story.append(Spacer(1, 2.0 * inch))
story.append(HRFlowable(width='80%', thickness=3, color=brand_primary, spaceAfter=12))
story.append(Paragraph('Complete Chart Gallery', styles['BrandTitle']))
story.append(Paragraph('Every ChartGenerator chart type rendered at 200 DPI', styles['BrandSubtitle']))
story.append(HRFlowable(width='80%', thickness=1, color=brand_secondary, spaceBefore=8, spaceAfter=24))
story.append(Paragraph('Prepared by: siege_utilities NB06', styles['BrandMeta']))
story.append(Spacer(1, 6))
story.append(Paragraph('Date: 2026', styles['BrandMeta']))

# Helper: one chart per page with branded heading and caption
def add_chart_page(story, heading, chart_img, caption):
    story.append(PageBreak())
    story.append(HRFlowable(width='100%', thickness=2, color=brand_primary, spaceAfter=8))
    story.append(Paragraph(heading, styles['BrandHeading']))
    story.append(Spacer(1, 8))
    story.extend(chart_gen_pdf.create_chart_with_caption(chart_img, caption))
    story.append(Spacer(1, 6))
    story.append(HRFlowable(width='40%', thickness=0.5, color=brand_secondary))

# =====================================================================
# 1. Bar Chart — Vertical
# =====================================================================
add_chart_page(story, '1a. Bar Chart (Vertical)',
    chart_gen_pdf.create_bar_chart(
        sales_data, x_column='Month', y_column='Revenue',
        title='Monthly Revenue',
    ),
    'Figure 1a: Vertical bar chart — monthly revenue.',
)

# =====================================================================
# 1b. Bar Chart — Horizontal
# =====================================================================
add_chart_page(story, '1b. Bar Chart (Horizontal)',
    chart_gen_pdf.create_bar_chart(
        sales_data, x_column='Month', y_column='Revenue',
        title='Monthly Revenue (Horizontal)', chart_type='horizontal',
    ),
    'Figure 1b: Horizontal bar chart — same data, rotated layout.',
)

# =====================================================================
# 2. Line Chart
# =====================================================================
add_chart_page(story, '2. Line Chart',
    chart_gen_pdf.create_line_chart(
        sales_data, x_column='Month', y_columns=['Revenue', 'Expenses'],
        title='Revenue vs Expenses',
    ),
    'Figure 2: Multi-series line chart comparing revenue and expenses.',
)

# =====================================================================
# 3. Pie Chart
# =====================================================================
add_chart_page(story, '3. Pie Chart',
    chart_gen_pdf.create_pie_chart(
        regional_data, labels_column='Region', values_column='Sales',
        title='Sales by Region',
    ),
    'Figure 3: Pie chart with legend table showing regional sales distribution.',
)

# =====================================================================
# 4. Scatter Plot
# =====================================================================
add_chart_page(story, '4. Scatter Plot',
    chart_gen_pdf.create_scatter_plot(
        correlation_data, x_column='x', y_column='y', color_column='z',
        title='Scatter with Color Coding',
    ),
    'Figure 4: Scatter plot with continuous color coding by z-value.',
)

# =====================================================================
# 5. Heatmap (Correlation Matrix)
# =====================================================================
add_chart_page(story, '5. Correlation Heatmap',
    chart_gen_pdf.create_heatmap(
        correlation_data, title='Correlation Matrix',
    ),
    'Figure 5: Seaborn heatmap showing pairwise correlations.',
)

# =====================================================================
# 6. Advanced Choropleth (matplotlib)
# =====================================================================
add_chart_page(story, '6. Advanced Choropleth',
    chart_gen_pdf.create_advanced_choropleth(
        ca_counties, geodata=ca_counties,
        location_column='geoid', value_column='population',
        title='CA Population by County (Quantiles)',
        classification='quantiles', bins=5,
    ),
    'Figure 6: Quantile-classified choropleth of California county populations.',
)

# =====================================================================
# 7. Bivariate Choropleth (matplotlib)
# =====================================================================
add_chart_page(story, '7. Bivariate Choropleth',
    chart_gen_pdf.create_bivariate_choropleth_matplotlib(
        ca_counties, geodata=ca_counties,
        location_column='geoid',
        value_column1='population', value_column2='median_income',
        title='Population vs Median Income',
    ),
    'Figure 7: Bivariate choropleth — population vs median income with 3x3 color grid.',
)

# =====================================================================
# 8. 3D Point Cloud
# =====================================================================
add_chart_page(story, '8. 3D Visualization',
    chart_gen_pdf.create_3d_map(
        point_data, latitude_column='lat', longitude_column='lon',
        elevation_column='value', title='3D US City Point Cloud',
    ),
    'Figure 8: 3D point cloud — US cities with elevation proportional to value.',
)

# =====================================================================
# 9. Dashboard (2x2)
# =====================================================================
add_chart_page(story, '9. Dashboard',
    chart_gen_pdf.create_dashboard(dashboard_charts, layout='2x2'),
    'Figure 9: Four-panel dashboard — bar, line, pie, and scatter.',
)

# =====================================================================
# 10. Summary Histograms
# =====================================================================
add_chart_page(story, '10. Distribution Summary',
    chart_gen_pdf.create_dataframe_summary_charts(
        correlation_data, title='Data Distributions',
    ),
    'Figure 10: Auto-generated histograms for all numeric columns.',
)

# =====================================================================
# Build PDF
# =====================================================================
template.build_document(story)

import os
pdf_size_kb = os.path.getsize(output_pdf) / 1024
su.log_info(f"PDF written to {output_pdf} ({pdf_size_kb:.0f} KB)")

## 9. PowerPoint Generation

In [ ]:
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt, Emu
    from pptx.dml.color import RGBColor
    from pptx.enum.text import PP_ALIGN
    from datetime import datetime

    prs = Presentation()
    prs.slide_width = Inches(13.333)   # widescreen 16:9
    prs.slide_height = Inches(7.5)

    # Branding colors for PPTX
    pptx_primary = RGBColor(0x1D, 0x36, 0x5D)    # #1D365D
    pptx_secondary = RGBColor(0x4A, 0x90, 0xE2)   # #4A90E2
    pptx_accent = RGBColor(0xFF, 0x6B, 0x35)       # #FF6B35
    pptx_white = RGBColor(0xFF, 0xFF, 0xFF)
    pptx_gray = RGBColor(0x66, 0x66, 0x66)

    # --- Helper: add branded title slide ---
    def add_title_slide(prs, title, subtitle=''):
        slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank layout
        # Navy background
        bg = slide.background
        fill = bg.fill
        fill.solid()
        fill.fore_color.rgb = pptx_primary
        # Title
        txBox = slide.shapes.add_textbox(Inches(1), Inches(2.0), Inches(11.3), Inches(1.5))
        tf = txBox.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(40)
        p.font.bold = True
        p.font.color.rgb = pptx_white
        p.alignment = PP_ALIGN.CENTER
        # Subtitle
        if subtitle:
            p2 = tf.add_paragraph()
            p2.text = subtitle
            p2.font.size = Pt(18)
            p2.font.color.rgb = pptx_secondary
            p2.alignment = PP_ALIGN.CENTER
        # Accent line
        from pptx.shapes.autoshape import Shape
        line = slide.shapes.add_shape(
            1, Inches(3), Inches(3.8), Inches(7.3), Pt(3),  # 1 = rectangle
        )
        line.fill.solid()
        line.fill.fore_color.rgb = pptx_accent
        line.line.fill.background()
        return slide

    # --- Helper: add chart image slide ---
    def add_chart_slide(prs, title, image_path, caption=''):
        slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
        # Title bar at top
        bar = slide.shapes.add_shape(1, Inches(0), Inches(0), prs.slide_width, Inches(0.9))
        bar.fill.solid()
        bar.fill.fore_color.rgb = pptx_primary
        bar.line.fill.background()
        txBox = slide.shapes.add_textbox(Inches(0.5), Inches(0.1), Inches(12), Inches(0.7))
        tf = txBox.text_frame
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(24)
        p.font.bold = True
        p.font.color.rgb = pptx_white
        # Chart image — centered
        img_path = str(image_path)
        pic = slide.shapes.add_picture(img_path, Inches(1.5), Inches(1.2), Inches(10.3), Inches(5.5))
        # Caption
        if caption:
            txBox2 = slide.shapes.add_textbox(Inches(1.5), Inches(6.9), Inches(10.3), Inches(0.4))
            tf2 = txBox2.text_frame
            p2 = tf2.paragraphs[0]
            p2.text = caption
            p2.font.size = Pt(10)
            p2.font.italic = True
            p2.font.color.rgb = pptx_gray
            p2.alignment = PP_ALIGN.LEFT
        return slide

    # --- Helper: add data table slide ---
    def add_table_slide(prs, title, df):
        slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
        # Title bar
        bar = slide.shapes.add_shape(1, Inches(0), Inches(0), prs.slide_width, Inches(0.9))
        bar.fill.solid()
        bar.fill.fore_color.rgb = pptx_primary
        bar.line.fill.background()
        txBox = slide.shapes.add_textbox(Inches(0.5), Inches(0.1), Inches(12), Inches(0.7))
        tf = txBox.text_frame
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(24)
        p.font.bold = True
        p.font.color.rgb = pptx_white
        # Table
        rows, cols = len(df) + 1, len(df.columns)
        tbl = slide.shapes.add_table(rows, cols, Inches(0.8), Inches(1.3), Inches(11.7), Inches(0.5 * rows)).table
        # Header row
        for i, col_name in enumerate(df.columns):
            cell = tbl.cell(0, i)
            cell.text = str(col_name)
            for paragraph in cell.text_frame.paragraphs:
                paragraph.font.size = Pt(11)
                paragraph.font.bold = True
                paragraph.font.color.rgb = pptx_white
            cell.fill.solid()
            cell.fill.fore_color.rgb = pptx_primary
        # Data rows
        for row_idx, (_, row) in enumerate(df.iterrows()):
            for col_idx, value in enumerate(row):
                cell = tbl.cell(row_idx + 1, col_idx)
                cell.text = str(value)
                for paragraph in cell.text_frame.paragraphs:
                    paragraph.font.size = Pt(10)
                # Alternate row shading
                if row_idx % 2 == 0:
                    cell.fill.solid()
                    cell.fill.fore_color.rgb = RGBColor(0xF6, 0xF6, 0xF6)
        return slide

    # =================================================================
    # Build the deck
    # =================================================================

    # 1. Title slide
    add_title_slide(prs,
        'Complete Chart Gallery',
        f'siege_utilities ChartGenerator | {datetime.now().strftime("%Y-%m-%d")}',
    )

    # 2. Sales data table
    add_table_slide(prs, 'Sales Summary — H1 2026', sales_data)

    # 3-13. Chart slides — one per saved PNG
    chart_slides = [
        ('Bar Chart (Vertical)',     OUTPUT_DIR / 'bar_vertical.png',        'Vertical bar chart — monthly revenue'),
        ('Bar Chart (Horizontal)',   OUTPUT_DIR / 'bar_horizontal.png',      'Horizontal bar chart — same data, rotated layout'),
        ('Line Chart',               OUTPUT_DIR / 'line_revenue_expenses.png','Multi-series line chart — revenue vs expenses'),
        ('Pie Chart',                OUTPUT_DIR / 'pie_regional_sales.png',  'Pie chart with legend table — regional sales'),
        ('Scatter Plot',             OUTPUT_DIR / 'scatter_color_coded.png', 'Scatter plot with color coding by z-value'),
        ('Correlation Heatmap',      OUTPUT_DIR / 'heatmap_correlation.png', 'Seaborn heatmap — pairwise correlations'),
        ('Advanced Choropleth',      OUTPUT_DIR / 'choropleth_quantiles.png','Quantile-classified choropleth — CA counties'),
        ('Bivariate Choropleth',     OUTPUT_DIR / 'bivariate_matplotlib.png','Bivariate choropleth — population vs income'),
        ('3D Visualization',         OUTPUT_DIR / '3d_point_cloud.png',     '3D point cloud — US cities'),
        ('Dashboard (2x2)',          OUTPUT_DIR / 'dashboard_2x2.png',      'Four-panel dashboard — bar, line, pie, scatter'),
        ('Distribution Summary',     OUTPUT_DIR / 'summary_histograms.png', 'Auto-generated histograms — all numeric columns'),
    ]

    added = 0
    for title, img_path, caption in chart_slides:
        if img_path.exists():
            add_chart_slide(prs, title, img_path, caption)
            added += 1
        else:
            su.log_warning(f"Skipping PPTX slide '{title}' — {img_path.name} not found")

    pptx_path = str(OUTPUT_DIR / 'sales_presentation.pptx')
    prs.save(pptx_path)
    pptx_kb = Path(pptx_path).stat().st_size / 1024
    su.log_info(f"PowerPoint created: {pptx_path} ({pptx_kb:.0f} KB, {len(prs.slides)} slides — 1 title + 1 table + {added} charts)")

except ImportError:
    su.log_warning("python-pptx not installed. Run: pip install python-pptx")

## 10. Chart Type Registry

The `ChartTypeRegistry` provides an extensible catalogue of chart configurations.
Each `ChartType` dataclass describes required/optional parameters, rendering defaults,
and feature flags (interactive, 3D, animation). The registry ships with built-in
types for geographic, statistical, temporal, and comparative charts, and supports
custom registrations at runtime.

In [ ]:
# 10. Chart Type Registry
# -----------------------------------------------------------------------
# Demonstrates ChartTypeRegistry: listing built-in chart types, inspecting
# categories, looking up individual types, and registering custom entries.
# -----------------------------------------------------------------------

try:
    from siege_utilities.reporting.chart_types import ChartTypeRegistry, ChartType

    # --- Instantiate a fresh registry (comes pre-loaded with defaults) ---
    registry = ChartTypeRegistry()

    # --- List all registered chart type names ---
    all_types = registry.list_chart_types()
    su.log_info(f"Total registered chart types: {len(all_types)}")
    for name in all_types:
        ct = registry.get_chart_type(name)
        su.log_info(f"  {name:35s}  category={ct.category}")

    # --- List available categories ---
    categories = registry.get_chart_categories()
    su.log_info(f"\nChart categories: {categories}")

    # --- Filter by category ---
    for cat in categories:
        cat_types = registry.list_chart_types(category=cat)
        su.log_info(f"  {cat:15s}: {cat_types}")

    # --- Look up a specific chart type and inspect its help dict ---
    help_info = registry.get_chart_help('bivariate_choropleth')
    su.log_info(f"\nHelp for 'bivariate_choropleth':")
    for key, value in help_info.items():
        su.log_info(f"  {key}: {value}")

    # --- Register a custom chart type ---
    custom_chart = ChartType(
        name='funnel_chart',
        category='comparative',
        description='Funnel chart for conversion pipeline visualization',
        required_parameters=['data', 'stage_column', 'value_column'],
        optional_parameters={
            'title': '',
            'color_scheme': 'blues',
            'show_percentages': True,
        },
        supports_interactive=False,
        supports_3d=False,
        default_width=8.0,
        default_height=6.0,
        custom_options={
            'orientation': 'vertical',
            'gap': 0.02,
        },
    )
    registry.register_chart_type(custom_chart)
    su.log_info(f"\nAfter registration — total chart types: {len(registry.list_chart_types())}")
    su.log_info(f"'funnel_chart' in registry: {registry.get_chart_type('funnel_chart') is not None}")

    # --- Validate parameters for a chart type ---
    valid = registry.validate_chart_parameters(
        'bar_chart',
        data=sales_data, x_column='Month', y_column='Revenue',
    )
    su.log_info(f"\nParameter validation for bar_chart (valid args): {valid}")

    missing_valid = registry.validate_chart_parameters(
        'bar_chart',
        data=sales_data,  # missing x_column, y_column
    )
    su.log_info(f"Parameter validation for bar_chart (missing args): {missing_valid}")

except ImportError as exc:
    su.log_warning(f"ChartTypeRegistry not available: {exc}")

## 11. Polling / Analytics

`PollingAnalyzer` provides cross-tabulation, longitudinal analysis, and performance
ranking capabilities designed for multi-dimensional analytics data (web analytics,
survey responses, campaign metrics, etc.).

Key methods demonstrated below:
- `create_cross_tabulation_matrix()` — all pairwise dimension cross-tabs
- `create_longitudinal_analysis()` — time-aggregated trends per dimension
- `create_performance_rankings()` — ranked dimension values by metric

In [ ]:
# 11. Polling / Analytics
# -----------------------------------------------------------------------
# Demonstrates PollingAnalyzer with synthetic analytics data:
#   - Cross-tabulation matrices across dimensions
#   - Longitudinal (time-series) analysis at multiple granularities
#   - Performance rankings by dimension
# -----------------------------------------------------------------------

try:
    from siege_utilities.reporting.analytics.polling_analyzer import PollingAnalyzer

    analyzer = PollingAnalyzer()
    su.log_info("PollingAnalyzer initialized")

    # --- Build synthetic analytics data ---
    np.random.seed(99)
    n_rows = 500
    polling_data = pd.DataFrame({
        'date': pd.date_range('2025-01-01', periods=n_rows, freq='D'),
        'channel': np.random.choice(['organic', 'paid', 'social', 'email', 'referral'], n_rows),
        'region': np.random.choice(['North', 'South', 'East', 'West', 'Central'], n_rows),
        'device': np.random.choice(['mobile', 'desktop', 'tablet'], n_rows, p=[0.6, 0.3, 0.1]),
        'sessions': np.random.randint(50, 2000, n_rows),
    })
    su.log_info(f"Synthetic polling data: {polling_data.shape}")
    display(polling_data.head())

    # ------------------------------------------------------------------
    # 11a. Cross-Tabulation Matrix
    # ------------------------------------------------------------------
    su.log_info("\n--- Cross-Tabulation Matrix ---")
    dimensions = ['channel', 'region', 'device']
    cross_tabs = analyzer.create_cross_tabulation_matrix(
        data=polling_data,
        dimensions=dimensions,
        metric='sessions',
        top_n=5,
    )
    for combo_name, ct_df in cross_tabs.items():
        su.log_info(f"\n  {combo_name}  (shape {ct_df.shape}):")
        display(ct_df)

    # ------------------------------------------------------------------
    # 11b. Longitudinal Analysis
    # ------------------------------------------------------------------
    su.log_info("\n--- Longitudinal Analysis ---")
    longitudinal = analyzer.create_longitudinal_analysis(
        data=polling_data.copy(),  # copy because method mutates 'period' column
        time_column='date',
        dimensions=['channel', 'region'],
        metric='sessions',
        periods=['weekly', 'monthly'],
    )
    for period, dim_dict in longitudinal.items():
        for dim_name, dim_df in dim_dict.items():
            su.log_info(f"  {period} x {dim_name}: {len(dim_df)} rows")
            display(dim_df.head())

    # ------------------------------------------------------------------
    # 11c. Performance Rankings
    # ------------------------------------------------------------------
    su.log_info("\n--- Performance Rankings ---")
    rankings = analyzer.create_performance_rankings(
        data=polling_data,
        dimensions=['channel', 'region', 'device'],
        metric='sessions',
        top_n=5,
    )
    for dim_name, ranked_items in rankings.items():
        su.log_info(f"\n  Top performers — {dim_name}:")
        for item_name, total, pct, growth in ranked_items:
            su.log_info(f"    {item_name:12s}  total={total:>8,}  share={pct:5.1f}%  growth={growth:+.1f}%")

    # ------------------------------------------------------------------
    # 11d. Polling Summary
    # ------------------------------------------------------------------
    su.log_info("\n--- Executive Summary ---")
    summary_input = {
        'channel': polling_data.groupby('channel')['sessions'].sum().reset_index(),
        'region': polling_data.groupby('region')['sessions'].sum().reset_index(),
    }
    summary_text = analyzer.create_polling_summary(summary_input, metric='sessions')
    su.log_info(summary_text)

except ImportError as exc:
    su.log_warning(f"PollingAnalyzer not available: {exc}")
except Exception as exc:
    su.log_error(f"PollingAnalyzer demo error: {exc}")
    import traceback
    traceback.print_exc()

<cell_type>markdown</cell_type>## Method Reference

Complete inventory of `ChartGenerator.create_*` methods and reporting components — all demonstrated above.

| # | Method | Backend | Returns | Section |
|---|--------|---------|---------|--------|
| 1 | `create_bar_chart` | matplotlib | RL Image | 2a |
| 2 | `create_line_chart` | matplotlib | RL Image | 2b |
| 3 | `create_pie_chart` | matplotlib | RL Image | 2c |
| 4 | `create_scatter_plot` | matplotlib | RL Image | 2d |
| 5 | `create_heatmap` | seaborn | RL Image | 2e |
| 6 | `create_choropleth_map` | Folium | Placeholder | 4a |
| 7 | `create_bivariate_choropleth` | Folium + gpd | Placeholder | 4b |
| 8 | `create_bivariate_choropleth_matplotlib` | matplotlib + gpd | RL Image | 4d |
| 9 | `create_advanced_choropleth` | matplotlib + gpd | RL Image | 4c |
| 10 | `create_marker_map` | Folium | Placeholder | 3a |
| 11 | `create_3d_map` | matplotlib 3D | RL Image | 5a |
| 12 | `create_heatmap_map` | Folium | Placeholder | 3b |
| 13 | `create_cluster_map` | Folium | Placeholder | 3c |
| 14 | `create_flow_map` | Folium | Placeholder | 3d |
| 15 | `create_proportional_text_bar_chart` | text | HTML string | 6a |
| 16 | `create_heatmap_text_chart` | text | HTML string | 6b |
| 17 | `create_dashboard` | matplotlib | RL Image | 7a |
| 18 | `create_dataframe_summary_charts` | matplotlib | RL Image | 7b |
| 19 | `create_chart_with_caption` | ReportLab | List[flowable] | 8 |
| 20 | `create_chart_section` | ReportLab | List[flowable] | 8 |
| 21 | `create_custom_chart` | dispatcher | RL Image | 7c |
| -- | `generate_chart_from_dataframe` | dispatcher | RL Image | 7d |

### Chart Type Registry (Section 10)

| Class | Module | Purpose |
|-------|--------|---------|
| `ChartTypeRegistry` | `siege_utilities.reporting.chart_types` | Extensible registry of chart type definitions |
| `ChartType` | `siege_utilities.reporting.chart_types` | Dataclass describing one chart type's params, defaults, and feature flags |

Key methods: `list_chart_types()`, `get_chart_type()`, `register_chart_type()`, `get_chart_categories()`, `get_chart_help()`, `validate_chart_parameters()`, `create_chart()`, `export_chart_type_config()`

### Polling / Analytics (Section 11)

| Class | Module | Purpose |
|-------|--------|---------|
| `PollingAnalyzer` | `siege_utilities.reporting.analytics.polling_analyzer` | Cross-tabulation, longitudinal, and ranking analytics |

Key methods: `create_cross_tabulation_matrix()`, `create_longitudinal_analysis()`, `create_performance_rankings()`, `create_polling_summary()`, `create_heatmap_visualization()`, `create_trend_analysis_chart()`

**Notes:**
- Methods 19 & 20 (`create_chart_with_caption`, `create_chart_section`) return ReportLab flowable lists — only usable inside a PDF build pipeline (Section 8).

**Files Generated** (all under `OUTPUT_DIR = output/notebook_06/`):
- `bar_vertical.png`, `bar_horizontal.png` — Bar charts
- `line_revenue_expenses.png` — Line chart
- `pie_regional_sales.png` — Pie chart
- `scatter_color_coded.png` — Scatter plot
- `heatmap_correlation.png` — Correlation heatmap
- `choropleth_quantiles.png` — Advanced choropleth (matplotlib)
- `bivariate_matplotlib.png` — Bivariate choropleth (matplotlib)
- `3d_point_cloud.png` — 3D point cloud
- `dashboard_2x2.png` — 2x2 dashboard
- `summary_histograms.png` — Summary histograms
- `custom_bar.png` — Custom dispatcher chart
- `dispatch_bar.png`, `dispatch_line.png`, `dispatch_pie.png`, `dispatch_scatter.png`, `dispatch_heatmap.png` — Dispatched charts
- `complete_chart_gallery.pdf` — Multi-page PDF with captioned charts
- `sales_presentation.pptx` — PowerPoint deck
- Folium HTML maps (marker, heatmap, cluster, flow, choropleth, bivariate)